# E-Commerce Sales & Customer Analytics

## 02 — Data Cleaning

### Objective

The objective of this notebook is to clean and prepare the raw
e-commerce datasets for exploratory data analysis and business
analysis.

The cleaning process includes:

- Converting data types
- Investigating missing values
- Handling duplicate records
- Validating identifiers
- Checking data consistency
- Creating useful analytical variables
- Preparing clean datasets for the next stage of the project

### 2. Import Libraries

In [1]:
import pandas as pd
import numpy as np

### 3. Load Raw Datasets

In [2]:
data_path = "../data/raw/"

customers = pd.read_csv(data_path + "olist_customers_dataset.csv")
orders = pd.read_csv(data_path + "olist_orders_dataset.csv")
order_items = pd.read_csv(data_path + "olist_order_items_dataset.csv")
products = pd.read_csv(data_path + "olist_products_dataset.csv")
payments = pd.read_csv(data_path + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(data_path + "olist_order_reviews_dataset.csv")
sellers = pd.read_csv(data_path + "olist_sellers_dataset.csv")

### 4. Create Working Copies

In [3]:
customers_clean = customers.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()
products_clean = products.copy()
payments_clean = payments.copy()
reviews_clean = reviews.copy()
sellers_clean = sellers.copy()

### 5. Data Type Conversion

In [4]:
order_date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in order_date_columns:
    orders_clean[col] = pd.to_datetime(
        orders_clean[col],
        errors="coerce"
    )

In [5]:
print(orders_clean[order_date_columns].dtypes)

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [6]:
review_date_columns = [
    'review_creation_date',
    'review_answer_timestamp'
]

for col in review_date_columns:
    reviews_clean[col] = pd.to_datetime(
        reviews_clean[col],
        errors='coerce'
    )

reviews_clean[review_date_columns].dtypes


review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

In [7]:
order_items_clean.dtypes

order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

In [8]:
order_items_clean['shipping_limit_date'] = pd.to_datetime(
    order_items_clean['shipping_limit_date'],
    errors='coerce'
)

In [9]:
order_items_clean.dtypes

order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

### 6. Missing Value Treatment

### Review Comments

Missing review titles and messages will not be treated as invalid
records. Customers can provide a rating without writing a textual
comment.

Therefore, these missing values will be retained rather than
removing the corresponding reviews.

### 6.2 Products

In [10]:
products_clean[
    products_clean['product_category_name'].isna()
].shape

(610, 9)

In [11]:
products_clean[
    products_clean['product_category_name'].isna()
].head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0


In [12]:
products_clean['product_category_name'] = (
    products_clean['product_category_name']
    .fillna('unknown')
)

In [13]:
products_clean['product_name_lenght'] = (
    products_clean['product_name_lenght']
    .fillna(0)
)

products_clean['product_description_lenght'] = (
    products_clean['product_description_lenght']
    .fillna(0)
)

products_clean['product_photos_qty'] = (
    products_clean['product_photos_qty']
    .fillna(0)
)


In [14]:
products_clean[
    [
        'product_category_name',
        'product_name_lenght',
        'product_description_lenght',
        'product_photos_qty'
    ]
].isnull().sum()

product_category_name         0
product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
dtype: int64

In [40]:
product_measurement_cols = [
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

for col in product_measurement_cols:
    products_clean[col] = pd.to_numeric(
        products_clean[col],
        errors='coerce'
    )

In [41]:
products_clean[
    product_measurement_cols
].isnull().sum()

product_weight_g     2
product_length_cm    2
product_height_cm    2
product_width_cm     2
dtype: int64

### Treatment of Missing Product Attributes

The 610 products missing category information will be retained rather
than removed. Their category will be labeled as `unknown` so that
these products remain available for other analyses.

Missing values in product name length, description length, and photo
quantity will be replaced with 0 because these fields represent
measurable product attributes and the missing records do not provide
a value for these attributes.

### 6.3 Orders

In [15]:
orders_clean[
    orders_clean['order_approved_at'].isna()
]['order_status'].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

In [16]:
orders_clean[
    orders_clean['order_delivered_carrier_date'].isna()
]['order_status'].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

In [17]:
orders_clean[
    orders_clean['order_delivered_customer_date'].isna()
]['order_status'].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [37]:
orders_clean['delivery_days'] = pd.to_numeric(
    orders_clean['delivery_days'],
    errors='coerce'
)

orders_clean['delivery_delay_days'] = pd.to_numeric(
    orders_clean['delivery_delay_days'],
    errors='coerce'
)

In [39]:
orders_clean[
    [
        'delivery_days',
        'delivery_delay_days'
    ]
].isnull().sum()

delivery_days          2965
delivery_delay_days    2965
dtype: int64

### 7. Duplicate & Identifier Checks

In [18]:
clean_datasets = {
    'customers': customers_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
    'products': products_clean,
    'payments': payments_clean,
    'reviews': reviews_clean,
    'sellers': sellers_clean
}


for name, df in clean_datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicate rows")

customers: 0 duplicate rows
orders: 0 duplicate rows
order_items: 0 duplicate rows
products: 0 duplicate rows
payments: 0 duplicate rows
reviews: 0 duplicate rows
sellers: 0 duplicate rows


In [19]:
id_summary = pd.DataFrame({
    'Dataset': [
        'Customers',
        'Orders',
        'Order Items',
        'Products',
        'Payments',
        'Reviews',
        'Sellers'
    ],
    'Rows': [
        len(customers_clean),
        len(orders_clean),
        len(order_items_clean),
        len(products_clean),
        len(payments_clean),
        len(reviews_clean),
        len(sellers_clean)
    ],
    'Unique IDs': [
        customers_clean['customer_id'].nunique(),
        orders_clean['order_id'].nunique(),
        order_items_clean['order_id'].nunique(),
        products_clean['product_id'].nunique(),
        payments_clean['order_id'].nunique(),
        reviews_clean['review_id'].nunique(),
        sellers_clean['seller_id'].nunique()
    ]
})

id_summary

,Dataset,Rows,Unique IDs
0,Customers,99441,99441
1,Orders,99441,99441
2,Order Items,112650,98666
3,Products,32951,32951
4,Payments,103886,99440
5,Reviews,99224,98410
6,Sellers,3095,3095


In [42]:
# Check total rows vs unique review IDs
print("Total rows:", len(reviews_clean))
print("Unique review IDs:", reviews_clean['review_id'].nunique())
print("Duplicate review IDs:", reviews_clean['review_id'].duplicated().sum())

Total rows: 99224
Unique review IDs: 98410
Duplicate review IDs: 814


In [43]:
duplicate_reviews = reviews_clean[
    reviews_clean['review_id'].duplicated(keep=False)
].sort_values('review_id')

duplicate_reviews

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07,2018-03-08 03:00:53
...,...,...,...,...,...,...,...
31120,fe5c833752953fed3209646f1f63b53c,4863e15fa53273cc7219c58f5ffda4fb,1,NaN,"Comprei dois produtos e ambos, mesmo enviados ...",2018-02-28,2018-02-28 13:57:52
7870,ff2fc9e68f8aabfbe18d710b83aabd30,2da58e0a7dcfa4ce1e00fad9d03ca3b5,2,NaN,NaN,2018-03-17,2018-03-19 11:44:15
82521,ff2fc9e68f8aabfbe18d710b83aabd30,1078d496cc6ab9a8e6f2be77abf5091b,2,NaN,NaN,2018-03-17,2018-03-19 11:44:15
73951,ffb8cff872a625632ac983eb1f88843c,c44883fc2529b4aa03ca90e7e09d95b6,3,NaN,NaN,2017-07-22,2017-07-26 13:41:07


### 8. Data Consistency Checks

In [20]:
reviews_clean['review_score'].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [21]:
reviews_clean['review_score'].min()

np.int64(1)

In [22]:
reviews_clean['review_score'].max()

np.int64(5)

The missing review text will be retained as NaN because the absence
of a comment does not make the review invalid. The review score can
still be used for analysis.

In [23]:
payments_clean['payment_value'].describe()

count    103886.000000
mean        154.100380
std         217.494064
min           0.000000
25%          56.790000
50%         100.000000
75%         171.837500
max       13664.080000
Name: payment_value, dtype: float64

In [24]:
payments_clean[
    payments_clean['payment_value'] < 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value


In [25]:
products_clean[
    (products_clean['product_weight_g'] <= 0) |
    (products_clean['product_length_cm'] <= 0) |
    (products_clean['product_height_cm'] <= 0) |
    (products_clean['product_width_cm'] <= 0)
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


### 9. Feature Engineering

In [26]:
orders_clean['delivery_days'] = (
    orders_clean['order_delivered_customer_date']
    - orders_clean['order_purchase_timestamp']
).dt.days

In [27]:
orders_clean['delivery_days'].describe()

count    96476.000000
mean        12.094086
std          9.551746
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: delivery_days, dtype: float64

In [28]:
orders_clean['delivery_delay_days'] = (
    orders_clean['order_delivered_customer_date']
    - orders_clean['order_estimated_delivery_date']
).dt.days

### Delivery Performance Variables

Two variables are created to support delivery-performance analysis:

- `delivery_days`: Number of days between order purchase and customer delivery.
- `delivery_delay_days`: Difference between actual delivery date and estimated delivery date.

A positive `delivery_delay_days` indicates that the order was delivered after the estimated delivery date.

A negative value indicates that the order was delivered before the estimated delivery date.

### 10. Final Data Quality Check

In [29]:
clean_datasets = {
    'customers': customers_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
    'products': products_clean,
    'payments': payments_clean,
    'reviews': reviews_clean,
    'sellers': sellers_clean
}

In [30]:
final_missing = pd.DataFrame()

for name, df in clean_datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    temp = pd.DataFrame({
        'Dataset': name,
        'Column': missing.index,
        'Missing Values': missing.values,
        'Missing %': (missing.values / len(df) * 100).round(2)
    })

    final_missing = pd.concat(
        [final_missing, temp],
        ignore_index=True
    )

final_missing

,Dataset,Column,Missing Values,Missing %
0,orders,order_approved_at,160,0.16
1,orders,order_delivered_carrier_date,1783,1.79
2,orders,order_delivered_customer_date,2965,2.98
3,orders,delivery_days,2965,2.98
4,orders,delivery_delay_days,2965,2.98
5,products,product_weight_g,2,0.01
6,products,product_length_cm,2,0.01
7,products,product_height_cm,2,0.01
8,products,product_width_cm,2,0.01
9,reviews,review_comment_title,87656,88.34


### Final Data Quality Assessment

The final missing-value report is used to verify that the cleaning
process was applied as intended.

Some missing values remain intentionally, particularly:

- Review titles and messages where customers did not provide written comments.
- Order delivery dates are missing for some orders, particularly
  orders that were canceled, unavailable, or had not reached the
  corresponding stage of the order lifecycle. A small number of
  delivered orders also have missing timestamps and will be retained
  for further investigation.

These values are retained because removing them would result in loss
of valid business information.

In [31]:
for name, df in clean_datasets.items():
    print(f"{name}: {df.shape}")

customers: (99441, 5)
orders: (99441, 10)
order_items: (112650, 7)
products: (32951, 9)
payments: (103886, 5)
reviews: (99224, 7)
sellers: (3095, 4)


In [32]:
for name, df in clean_datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicate rows")

customers: 0 duplicate rows
orders: 0 duplicate rows
order_items: 0 duplicate rows
products: 0 duplicate rows
payments: 0 duplicate rows
reviews: 0 duplicate rows
sellers: 0 duplicate rows


In [33]:
orders_clean['delivery_days'].describe()

count    96476.000000
mean        12.094086
std          9.551746
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: delivery_days, dtype: float64

In [34]:
orders_clean[
    orders_clean['delivery_days'] < 0
][[
    'order_id',
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'delivery_days'
]].head()

,order_id,order_purchase_timestamp,order_delivered_customer_date,delivery_days


In [46]:
orders_clean[
    orders_clean['delivery_delay_days'] > 0
]['delivery_delay_days'].describe()

count    6535.000000
mean       10.620352
std        14.643844
min         1.000000
25%         3.000000
50%         7.000000
75%        13.000000
max       188.000000
Name: delivery_delay_days, dtype: float64

### 11. Save Cleaned Data

In [49]:
processed_path = "../data/processed/"

customers_clean.to_csv(
    processed_path + "customers_clean.csv",
    index=False
)

orders_clean.to_csv(
    processed_path + "orders_clean.csv",
    index=False,
    na_rep='\\N',
    lineterminator='\n'
)

order_items_clean.to_csv(
    processed_path + "order_items_clean.csv",
    index=False
)

products_clean.to_csv(
    processed_path + "products_clean.csv",
    index=False,
    na_rep='\\N',
    lineterminator='\n'
)

payments_clean.to_csv(
    processed_path + "payments_clean.csv",
    index=False
)

reviews_clean.to_csv(
    processed_path + "reviews_clean.csv",
    index=False,
    na_rep='\\N',
    lineterminator='\n'
)

sellers_clean.to_csv(
    processed_path + "sellers_clean.csv",
    index=False
)

## 12. Cleaning Summary

The raw datasets were prepared for analysis by:

- Converting date columns to datetime format.
- Investigating missing values based on their business meaning.
- Retaining valid reviews with missing written comments.
- Handling missing product attributes appropriately.
- Validating duplicate records and identifiers.
- Checking numerical and categorical values for inconsistencies.
- Creating delivery-related analytical variables.
- Saving cleaned datasets separately from the raw data.